# OCR + Hybrid Model Evaluation Pipeline
End-to-end simulation: image → OCR → preprocessing → hybrid model → metrics → error analysis.

In [ ]:
python3 scripts/download_off_images.py --count 30
python3 scripts/run_ocr_simulation.py --dir data/ocr_test_images --out pipeline_results_ocr_sim.csv
# Or open and run [notebooks/06_ocr_hybrid_pipeline.ipynb] in Jupyter to produce pipeline_results_notebook_fixed.csv

## Overview
End-to-end pipeline: Image → OCR → Hybrid Model → Allergen Predictions.

## Workflow
1. Simulate image input
2. Apply OCR extraction
3. Run hybrid model
4. Generate allergen report
5. Evaluate accuracy


In [1]:
import os
import time
import ast
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
import easyocr

## Configuration

In [2]:
IMAGE_FOLDER = "data/ocr_test_images"
LABEL_FILE = "data/ocr_test_images/labels.csv" if os.path.exists('data/ocr_test_images/labels.csv') else None  # optional: CSV with columns [filename, true_labels]

ALLERGENS = ["milk","eggs","peanuts","tree_nuts","soy","wheat","fish","shellfish"]

## Load OCR

In [3]:
reader = easyocr.Reader(['en'], gpu=False)

Using CPU. Note: This module is much faster with a GPU.


## Model Hooks (Replace with your actual functions)

In [4]:
def preprocess_text(text):
    return text.lower().strip()

def hybrid_predict(texts):
    # TODO: replace with your actual hybrid model
    preds = [["milk"] if t else [] for t in texts]
    return preds, None

## OCR Extraction

In [5]:
def extract_text(image_path):
    try:
        results = reader.readtext(image_path, detail=0)
        return " ".join(results)
    except Exception:
        return ""

## Helpers

In [6]:
def labels_to_binary(label_list, classes):
    binary = np.zeros((len(label_list), len(classes)))
    for i, labels in enumerate(label_list):
        for l in labels:
            if l in classes:
                binary[i][classes.index(l)] = 1
    return binary

def safe_parse_list(x):
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

## Run Pipeline

In [7]:
def run_pipeline(image_folder):
    image_paths = [os.path.join(image_folder, f) for f in os.listdir(image_folder)
                   if f.lower().endswith(('.jpg','.jpeg','.png'))]

    texts, filenames, ocr_fail = [], [], []
    times = []

    print("Running OCR...")
    for path in tqdm(image_paths):
        start = time.time()
        text = extract_text(path)
        elapsed = time.time() - start

        processed = preprocess_text(text)

        texts.append(processed)
        filenames.append(os.path.basename(path))
        ocr_fail.append(1 if processed.strip()=="" else 0)
        times.append(elapsed)

    print("Running Hybrid Model...")
    preds, _ = hybrid_predict(texts)

    df = pd.DataFrame({
        "filename": filenames,
        "text": texts,
        "predictions": preds,
        "ocr_failed": ocr_fail,
        "ocr_time": times
    })

    return df

## Evaluation Metrics

In [8]:
def evaluate(df, label_file):
    labels_df = pd.read_csv(label_file)
    df = df.merge(labels_df, on="filename")

    df["true_labels"] = df["true_labels"].apply(safe_parse_list)

    y_true = labels_to_binary(df["true_labels"], ALLERGENS)
    y_pred = labels_to_binary(df["predictions"], ALLERGENS)

    print("=== Classification Report ===")
    print(classification_report(y_true, y_pred, target_names=ALLERGENS, zero_division=0))

    print("F1 Micro:", f1_score(y_true, y_pred, average="micro"))
    print("F1 Macro:", f1_score(y_true, y_pred, average="macro"))
    print("Precision:", precision_score(y_true, y_pred, average="micro"))
    print("Recall:", recall_score(y_true, y_pred, average="micro"))

    return df, y_true, y_pred

## Error Analysis

In [9]:
def error_analysis(df, y_true=None, y_pred=None):
    print("=== OCR Failures ===")
    print("Total OCR failures:", df["ocr_failed"].sum())

    print("\n=== OCR Time ===")
    print("Average OCR time:", df["ocr_time"].mean())

    if y_true is not None:
        print("\n=== Per-Class F1 ===")
        per_class_f1 = f1_score(y_true, y_pred, average=None)
        for cls, score in zip(ALLERGENS, per_class_f1):
            print(f"{cls}: {score:.3f}")

        print("\n=== Sample Errors ===")
        errors = []
        for i in range(len(df)):
            if not np.array_equal(y_true[i], y_pred[i]):
                errors.append(i)

        print("Total errors:", len(errors))

        for i in errors[:5]:
            print("\n---")
            print("File:", df.iloc[i]['filename'])
            print("Text:", df.iloc[i]['text'][:200])
            print("True:", df.iloc[i]['true_labels'])
            print("Pred:", df.iloc[i]['predictions'])

## Run Everything

In [10]:
df = run_pipeline(IMAGE_FOLDER)

if LABEL_FILE and os.path.exists(LABEL_FILE):
    df, y_true, y_pred = evaluate(df, LABEL_FILE)
    error_analysis(df, y_true, y_pred)
else:
    print("No labels provided or label file missing → running pipeline only")
    error_analysis(df)

df.to_csv("pipeline_results.csv", index=False)
print("Saved to pipeline_results.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'data/ocr_test_images'